In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [2]:
import pandas as pd

df = pd.read_csv("data.csv")

df["DATETIME"] = pd.to_datetime(df["DATETIME"])
df["mois"] = df["DATETIME"].dt.month
df["annee"] = df["DATETIME"].dt.year

df["GAMME"] = (
    df["GAMME"]
    .replace("#REF!", "ND")
    .str.replace("JEUX / ENFANTS", "JEUX_ENFANTS", regex=False)
    .fillna("ND")
)

gammes = df["GAMME"].unique()


df_gamme_dummies = pd.get_dummies(df["GAMME"], dtype=int)

df = pd.concat([df, df_gamme_dummies], axis=1)

# Montant de vente
df["montant_vente"] = df["PRIX TTC"] * df["QUANTITE"]

# Infos hôtel à garder
hotel_cols = [
    "HOTEL_NAME",
    "HOTEL_CITY",
    "HOTEL_LAT",
    "HOTEL_LON",
]

train_df = df[df["annee"] < 2026]

# Agrégation par hôtel / mois / type
monthly = (
    train_df
    .groupby(hotel_cols + ["mois", "TYPE", "GAMME"])
    .agg(
        montant=("montant_vente", "sum"),
        nbr_ventes=("ORDER ID (TICKET DE CAISSE)", "nunique"),
        # **{
        #     f"sum_GAMME_{gamme}": (gamme, "sum")
        #     for gamme in df_gamme_dummies
        # }
    )
    .reset_index()
)

# Passage en colonnes
wide = monthly.pivot_table(
    index=hotel_cols,
    columns=["mois", "TYPE", "GAMME"],
    values=["montant", "nbr_ventes"],
    fill_value=0
)

# Aplatir les noms de colonnes
wide.columns = [
    f"m{mois:02d}__{type_}__{gamme_}__{metric}"
    .replace("&", "")
    .replace("-", "_")
    .replace(" ", "_")
    for metric, mois, type_, gamme_ in wide.columns
]

df_hotel_monthly = wide.reset_index()

# Trier les colonnes : infos hôtel puis m01, m02, ...
hotel_cols_final = hotel_cols

monthly_cols = sorted(
    [c for c in df_hotel_monthly.columns if c not in hotel_cols_final],
    key=lambda x: (
        int(x.split("_")[0][1:]),  # mois
        x
    )
)

df_hotel_monthly = df_hotel_monthly[hotel_cols_final + monthly_cols]

In [3]:
df_hotel_monthly.head(10)

,HOTEL_NAME,HOTEL_CITY,HOTEL_LAT,HOTEL_LON,m01__FB__ALCOOL__montant,m01__FB__ALCOOL__nbr_ventes,m01__FB__FOOD_SALEE__montant,m01__FB__FOOD_SALEE__nbr_ventes,m01__FB__FOOD_SUCREE__montant,m01__FB__FOOD_SUCREE__nbr_ventes,m01__FB__SANS_ALCOOL__montant,m01__FB__SANS_ALCOOL__nbr_ventes,m01__FB__SOUVENIRS__montant,m01__FB__SOUVENIRS__nbr_ventes,m01__NON_FB__ACCESSOIRES__montant,m01__NON_FB__ACCESSOIRES__nbr_ventes,m01__NON_FB__COSMETIQUE__montant,m01__NON_FB__COSMETIQUE__nbr_ventes,m01__NON_FB__JEUX_ENFANTS__montant,m01__NON_FB__JEUX_ENFANTS__nbr_ventes,m01__NON_FB__ND__montant,m01__NON_FB__ND__nbr_ventes,m01__NON_FB__PAP__montant,m01__NON_FB__PAP__nbr_ventes,m01__NON_FB__SOS__montant,m01__NON_FB__SOS__nbr_ventes,m01__NON_FB__SOUVENIRS__montant,m01__NON_FB__SOUVENIRS__nbr_ventes,m02__FB__ALCOOL__montant,m02__FB__ALCOOL__nbr_ventes,m02__FB__FOOD_SALEE__montant,m02__FB__FOOD_SALEE__nbr_ventes,m02__FB__FOOD_SUCREE__montant,m02__FB__FOOD_SUCREE__nbr_ventes,m02__FB__SANS_ALCOOL__montant,m02__FB__SANS_ALCOOL__nbr_ventes,m02__FB__SOUVENIRS__montant,m02__FB__SOUVENIRS__nbr_ventes,m02__NON_FB__ACCESSOIRES__montant,m02__NON_FB__ACCESSOIRES__nbr_ventes,m02__NON_FB__COSMETIQUE__montant,m02__NON_FB__COSMETIQUE__nbr_ventes,m02__NON_FB__JEUX_ENFANTS__montant,m02__NON_FB__JEUX_ENFANTS__nbr_ventes,m02__NON_FB__ND__montant,m02__NON_FB__ND__nbr_ventes,m02__NON_FB__PAP__montant,m02__NON_FB__PAP__nbr_ventes,m02__NON_FB__SOS__montant,m02__NON_FB__SOS__nbr_ventes,m02__NON_FB__SOUVENIRS__montant,m02__NON_FB__SOUVENIRS__nbr_ventes,m03__FB__ALCOOL__montant,m03__FB__ALCOOL__nbr_ventes,m03__FB__FOOD_SALEE__montant,m03__FB__FOOD_SALEE__nbr_ventes,m03__FB__FOOD_SUCREE__montant,m03__FB__FOOD_SUCREE__nbr_ventes,m03__FB__SANS_ALCOOL__montant,m03__FB__SANS_ALCOOL__nbr_ventes,m03__FB__SOUVENIRS__montant,m03__FB__SOUVENIRS__nbr_ventes,m03__NON_FB__ACCESSOIRES__montant,m03__NON_FB__ACCESSOIRES__nbr_ventes,m03__NON_FB__COSMETIQUE__montant,m03__NON_FB__COSMETIQUE__nbr_ventes,m03__NON_FB__JEUX_ENFANTS__montant,m03__NON_FB__JEUX_ENFANTS__nbr_ventes,m03__NON_FB__ND__montant,m03__NON_FB__ND__nbr_ventes,m03__NON_FB__PAP__montant,m03__NON_FB__PAP__nbr_ventes,m03__NON_FB__SOS__montant,m03__NON_FB__SOS__nbr_ventes,m03__NON_FB__SOUVENIRS__montant,m03__NON_FB__SOUVENIRS__nbr_ventes,m04__FB__ALCOOL__montant,m04__FB__ALCOOL__nbr_ventes,m04__FB__FOOD_SALEE__montant,m04__FB__FOOD_SALEE__nbr_ventes,m04__FB__FOOD_SUCREE__montant,m04__FB__FOOD_SUCREE__nbr_ventes,m04__FB__SANS_ALCOOL__montant,m04__FB__SANS_ALCOOL__nbr_ventes,m04__FB__SOUVENIRS__montant,m04__FB__SOUVENIRS__nbr_ventes,m04__NON_FB__ACCESSOIRES__montant,m04__NON_FB__ACCESSOIRES__nbr_ventes,m04__NON_FB__COSMETIQUE__montant,m04__NON_FB__COSMETIQUE__nbr_ventes,m04__NON_FB__JEUX_ENFANTS__montant,m04__NON_FB__JEUX_ENFANTS__nbr_ventes,m04__NON_FB__ND__montant,m04__NON_FB__ND__nbr_ventes,m04__NON_FB__PAP__montant,m04__NON_FB__PAP__nbr_ventes,m04__NON_FB__SOS__montant,m04__NON_FB__SOS__nbr_ventes,m04__NON_FB__SOUVENIRS__montant,m04__NON_FB__SOUVENIRS__nbr_ventes,m05__FB__ALCOOL__montant,m05__FB__ALCOOL__nbr_ventes,m05__FB__FOOD_SALEE__montant,m05__FB__FOOD_SALEE__nbr_ventes,m05__FB__FOOD_SUCREE__montant,m05__FB__FOOD_SUCREE__nbr_ventes,m05__FB__SANS_ALCOOL__montant,m05__FB__SANS_ALCOOL__nbr_ventes,m05__FB__SOUVENIRS__montant,m05__FB__SOUVENIRS__nbr_ventes,m05__NON_FB__ACCESSOIRES__montant,m05__NON_FB__ACCESSOIRES__nbr_ventes,m05__NON_FB__COSMETIQUE__montant,m05__NON_FB__COSMETIQUE__nbr_ventes,m05__NON_FB__JEUX_ENFANTS__montant,m05__NON_FB__JEUX_ENFANTS__nbr_ventes,m05__NON_FB__ND__montant,m05__NON_FB__ND__nbr_ventes,m05__NON_FB__PAP__montant,m05__NON_FB__PAP__nbr_ventes,m05__NON_FB__SOS__montant,m05__NON_FB__SOS__nbr_ventes,m05__NON_FB__SOUVENIRS__montant,m05__NON_FB__SOUVENIRS__nbr_ventes,m06__FB__ALCOOL__montant,m06__FB__ALCOOL__nbr_ventes,m06__FB__FOOD_SALEE__montant,m06__FB__FOOD_SALEE__nbr_ventes,m06__FB__FOOD_SUCREE__montant,m06__FB__FOOD_SUCREE__nbr_ventes,m06__FB__SANS_ALCOOL__montant,m06__FB__SANS_ALCOOL__nbr_ventes,m06__FB

In [21]:
train_df.head(1)

,NOM BOUTIQUE,OPERATEUR,MACHINE,NOM DU PRODUIT,QUANTITE,VAT,PRIX TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER ID (TICKET DE CAISSE),DATETIME,DATE_HEURE,MOIS_D_ANNEE,JOUR_DU_MOIS,JOUR_DE_SEMAINE,JOUR_D_ANNEE,SEMAINE_D_ANNEE,IS_WEEKEND,HEURE_DU_JOUR,HOTEL_NAME,HOTEL_CITY,HOTEL_LAT,HOTEL_LON,ALCOHOL_1_KM,ALCOHOL_2_KM,ALCOHOL_3_KM,BAKERY_1_KM,BAKERY_2_KM,BAKERY_3_KM,CONFECTIONERY_2_KM,CONFECTIONERY_3_KM,CONVENIENCE_1_KM,CONVENIENCE_2_KM,CONVENIENCE_3_KM,COSMETICS_2_KM,COSMETICS_3_KM,GIFT_2_KM,GIFT_3_KM,KIOSK_1_KM,KIOSK_2_KM,KIOSK_3_KM,SUPERMARKET_1_KM,SUPERMARKET_2_KM,SUPERMARKET_3_KM,TOBACCO_1_KM,TOBACCO_2_KM,TOBACCO_3_KM,BEVERAGES_1_KM,BEVERAGES_2_KM,BEVERAGES_3_KM,CONFECTIONERY_1_KM,COSMETICS_1_KM,GIFT_1_KM,GROCERY_2_KM,GROCERY_3_KM,GROCERY_1_KM,ICE_CREAM_2_KM,ICE_CREAM_3_KM,time,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco,lat,lon,mois,annee,montant_vente
0,Ibis budget Nice,ADIPOS,SCANNER NICE,TONGS FEMME 100 NOIR,1,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9281,2023-08-10 12:04:22.555,2023-08-10 12:00:00,8,10,3,222,32,0,12,Ibis budget Nice,Nice,43.667571,7.214308,1,2,2,2,16,26,1.0,1.0,1,9,20,12.0,12.0,1,1,1.0,3.0,3.0,1.0,7,11,1.0,4.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-08-10 12:00:00,25.0,18.7,68.0,0.0,0.0,130.0,9.4,17.0,1018.7,55.0,2.0,43.667571,7.214308,8,2023,6.0


In [4]:
df_hotel_monthly.to_excel("transaction_prepared_data.xlsx", index = False)

In [52]:
train_df["HOTEL_NAME"].unique()

array(['Ibis budget Nice', 'Mercure Paris Montmartre Sacré-Cœur',
       'Novotel Megève Mont-Blanc', 'Novotel Paris Tour Eiffel',
       'Ibis budget Strasbourg Centre République'], dtype=object)